<a href="https://colab.research.google.com/github/elishamoore56-star/Elisha_INFO4670_Fall2026/blob/main/Elisha_Moore's_Copy_of_Week5_INFO4670_Assignment2_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [12]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [13]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
# Where are the blanks?
missing = students.isna().sum()
print (missing[missing > 0])
n_missing = students["study_hours_reported"].isna().sum()
print(f"\nstudy_hours_reported: {n_missing} blank of {len(students)} "
      f"({n_missing/len(students)*100:.1f}%) -> {students[ 'study_hours_reported'].notna().sum()} present")
n_missing_study = n_missing

study_hours_reported    255
dtype: int64

study_hours_reported: 255 blank of 2027 (12.6%) -> 1772 present


**Your justification (1–2 sentences):** _..._

In [14]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [15]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
print ("RAW housing - eight spellings for three groups:")
print (students["housing"].value_counts())

clean_map = {"on-campus":"On-Campus", "off-campus":"Off-Campus",
             "off campus":"Off-Campus", "with family":"With Family"}
students["housing_clean"] = students["housing"].str.strip().str.lower().map(clean_map)

print ("\nCLEANED - three real groups:")
print(students["housing_clean"].value_counts())

RAW housing - eight spellings for three groups:
housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64

CLEANED - three real groups:
housing_clean
Off-Campus     945
On-Campus      590
With Family    492
Name: count, dtype: int64


In [16]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [17]:
# @title
impossible_ages = sorted(int(a) for a in students.loc[(students["age"] < 15) | (students["age"] > 90), "age"].unique())
n_neg_work = (students["work_hours_per_week"] < 0).sum()
zeros = students["final_gpa"] == 0
print("\nGPA = 0.00 rows:", int(zeros.sum()),
      "| of those, marked dropped_out:", int(students.loc[zeros, "dropped_out"].sum()))


GPA = 0.00 rows: 43 | of those, marked dropped_out: 9


In [18]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [19]:
# TODO: build students_dedup (one row per student)
students_dedup = students.drop_duplicates(subset=['student_id'])
print("student_records rows:", len(students), "| unique student_id:", students["student_id"].nunique())
print("exact duplicate rows:", students.duplicated().sum(),
" | duplicate student_ids:", students["student_id"].duplicated().sum())
print(f"\nenroll : one row per enrollment -> {len(enroll)} rows for {enroll["sid"].nunique()} students")
print(f"activity: one row per student-week -> {len(activity)} rows for {activity["student_id"].nunique()} students x {activity["week"].nunique()} weeks")

student_records rows: 2027 | unique student_id: 2000
exact duplicate rows: 18  | duplicate student_ids: 27

enroll : one row per enrollment -> 8088 rows for 2014 students
activity: one row per student-week -> 32000 rows for 2000 students x 16 weeks


**Why not de-dupe enroll / activity? (1 sentence):** _..._

In [20]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [21]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
analysis = students_dedup.copy()

analysis["student_key_num"] = analysis["student_id"].str.replace("NU-", "").astype(int)

activity_summary = activity.groupby("student_id").agg(
    total_lms_clicks=('lms_clicks', 'sum'),
    total_minutes_active=('minutes_active', 'sum')
).reset_index()

analysis = analysis.merge(activity_summary, on="student_id", how="left")

In [22]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [14]:
# TODO
key = students["student_id"].str.replace("NU-", "", regex=False).astype(int)

matched    = enroll["sid"].isin(set(key)).sum()
orphan_sids = set(enroll["sid"]) - set(key)
print("enrollment rows that match a student:", matched, "of", len(enroll))
print("orphan IDs (no student record):", len(orphan_sids),
      "-> orphan rows:)", enroll["sid"].isin(orphan_sids).sum())


enrollment rows that match a student: 8041 of 8088
orphan IDs (no student record): 14 -> orphan rows:) 47


In [15]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [5]:
import re

def date_format(s):
  if re.match(r"^\d{4}-\d{2}$", s):     return "YYYY-MM-DD"
  if re.match(r"^\d{1,2}/\d{1,2}/\d{4}$", s):  return "M/D/YYYY"
  if re.match(r"^\d{1,2}-[A-Za-z]{3}-\d{4}$", s):  return "D-Mon_YYYY"

print(students["enrollment_date"].map(date_format).value_counts())

naive  = pd.to_datetime(students["enrollment_date"], errors="coerce")
deliberate = pd.to_datetime(students["enrollment_date"], format="mixed", errors="coerce")
print("\nnaive parse    -> blanks (NaT):", naive.isna().sum())
print("deliberate parse -> blanks (NaT):", deliberate.isna().sum())
dates_parsed = deliberate

enrollment_date
M/D/YYYY      1165
D-Mon_YYYY     305
Name: count, dtype: int64

naive parse    -> blanks (NaT): 1470
deliberate parse -> blanks (NaT): 0


In [17]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [21]:

analysis["study_z"] = (analysis["study_hours_reported"] - analysis["study_hours_reported"].mean()) / analysis["study_hours_reported"].std()


analysis["gpa_band"] = pd.cut(analysis["final_gpa"],
                             bins=[-0.01, 1, 2, 3, 4],
                             labels=["0.1", "1-2", "2-3", "3-4"])


print(f"Mean of analysis['study_z']: {analysis['study_z'].mean():.2f}")
print(f"Standard Deviation of analysis['study_z']: {analysis['study_z'].std():.2f}")
print("\nValue counts for analysis['gpa_band']:")
print(analysis["gpa_band"].value_counts().sort_index())

Mean of analysis['study_z']: 0.00
Standard Deviation of analysis['study_z']: 1.00

Value counts for analysis['gpa_band']:
gpa_band
0.1      70
1-2     528
2-3    1102
3-4     300
Name: count, dtype: int64


In [22]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [29]:
# TODO: write analysis to northgate_clean.csv
analysis.to_csv("northgate_clean.csv", index=False)
print("Wrote analysis to northgate_clean.csv")

Wrote analysis to northgate_clean.csv


In [30]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**
- _decision → justification_
- Missing Values -> study_hours_reported 255
- Inconsistent Categories -> 3 Groups: (On campus, off campus, With Family)
- Errors vs. extremes -> Impossible Ages
- Duplicates -> Checkin duplicates_drop
-Standardize the key -> One Row Per Student, 2000 rows
-Dates -> All dates are parsed

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [25]:
# Calculate the mean GPA
mean_gpa = analysis["final_gpa"].mean()
print(f"Mean GPA from the clean analysis table: {mean_gpa:0}")

# Explore a relationship: average GPA by GPA band
gpa_by_band = analysis.groupby("gpa_band")["final_gpa"].mean().sort_index()
print("\nAverage GPA by GPA Band:")
print(gpa_by_band)



Mean GPA from the clean analysis table: 2.3053600000000003


KeyError: 'gpa_band'

In [9]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
